# TheLook eCommerce - Project Walkthrough

**Commercial analytics on a 180,000-line retail dataset: a governed warehouse,
a plan-versus-actual layer fed by a second source system, and the controls that
stop a wrong number reaching a report.**

---

### How to read this

This notebook **does not build anything**. The pipeline builds; this reads. It
opens the finished warehouse read-only and walks it in the order it was
constructed, so the whole project can be followed in one pass instead of across
twenty files.

Every section has the same shape:

| | |
|---|---|
| **What it does** | the purpose of the layer, in a few sentences |
| **Built by** | the production file that owns it |
| **Written to** | where the output lands on disk |
| **Result** | a live query against the warehouse |
| **Conclusion** | what that result supports |

### Seeing the code, not a description of it

Where a section is about engineering rather than a result, the notebook prints the
**production code itself**. `show_sql("mart.plan_vs_actual_monthly")` reads that
statement out of `sql/duckdb/` while the cell runs; `show_py(...)` does the same for a
function in `src/`. Nothing is copied into this notebook, so what you read here and
what the pipeline executes cannot drift apart. Long statements are cut off with a line
count and the file name so the rest is one click away.

### Running it yourself

One command rebuilds everything from the seven source CSVs - warehouse, exports,
figures, Excel pack, notebooks and the validation gate:

```bash
python src/run_all.py --source-csv-dir <folder with the 7 CSVs> --rebuild
```

What that runs, in what order, how to run a single stage while developing, and why
the work is split across many files instead of one, is set out in
[`docs/how_to_run.md`](../docs/how_to_run.md).

### Where the exploring happens

This notebook is the finished account. The working surface is
[`notebooks/90_workbench.ipynb`](90_workbench.ipynb), backed by `src/explore.py` -
a read-only handle on the warehouse with the verbs you need before writing anything:
`cols()` for types and null counts, `grain()` for whether a key is unique, `overlap()`
for whether two keys actually match, and `tie()` for whether a total survived a layer.

### If you want to read the code

[`docs/code_map.md`](../docs/code_map.md) indexes **every object in the warehouse to
the file and line that builds it**, with the statement length beside it, so it is clear
which objects carry real logic and which are a projection. It is generated from the
files on every build, so it cannot go stale. The full source of any object can also be
printed on demand with `w.source("mart.price_volume_mix")` from the workbench.


### The project in one page

```
  7 source CSVs          2 reference seeds        1 plan submission file
  (TheLook export)       (country, FX)            (regional, local currency)
         |                      |                          |
         +----------------------+--------------------------+
                                |
                      [ raw ]   ingested exactly as delivered
                                |
                   SOURCE QUALITY GATE   runs before any model exists
                                |
                      [ stg ]   typed, renamed, harmonised
                                |
                      [ core ]  conformed dimensions and facts
                                |
                      [ mart ]  one table per business question
                                |
                  WAREHOUSE AND COMMERCIAL GATES
                                |
         +----------------------+--------------------------+
         |                      |                          |
   Power BI, 8 pages      CSV / Parquet exports      Excel management pack
```

| Folder | Holds |
|---|---|
| `src/` | the pipeline, the orchestrator, the Python analytics |
| `sql/duckdb/` | ten numbered SQL files, run in sequence |
| `seeds/` | small reference tables that are committed, not derived |
| `data/planning/` | the regional plan submission file - a second source |
| `artifacts/` | the warehouse, the figures, the run metadata |
| `outputs/` | QA evidence and the modelled exports |
| `docs/` | the written analysis and reference documents |
| `power_bi/` | the report and its DAX measures |


---

## Part 1 - Open the warehouse


In [1]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 70)
# Previews show at least 10 rows - 3 is not enough to see a pattern,
# a null cluster, or a grain that is finer than you assumed.
pd.set_option("display.max_rows", 40)
pd.set_option("display.min_rows", 20)

# Read-only on purpose. The pipeline owns the warehouse; nothing here writes to it.
CURRENT = Path.cwd()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
DB = ROOT / "artifacts" / "thelook_analytics.duckdb"
db = duckdb.connect(str(DB), read_only=True)


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the production warehouse and return a DataFrame."""
    return db.execute(sql).fetchdf()


SQL_DIR = ROOT / "sql" / "duckdb"


def show_sql(obj, lines=None):
    """Print the production SQL that builds `obj`.

    The text is read from the file the pipeline actually executes, so what
    is shown here and what runs cannot drift apart.
    """
    for path in sorted(SQL_DIR.glob("*.sql")):
        body = path.read_text(encoding="utf-8")
        for kind in ("TABLE", "VIEW"):
            marker = f"CREATE OR REPLACE {kind} {obj} AS"
            if marker not in body:
                continue
            statement = []
            for row in body[body.index(marker):].splitlines():
                statement.append(row)
                if row.rstrip().endswith(";"):
                    break
            shown = statement if lines is None else statement[:lines]
            print(f"--- sql/duckdb/{path.name} - {len(statement)} lines " + "-" * 22)
            print("\n".join(shown))
            if lines is not None and len(statement) > lines:
                print(f"... {len(statement) - lines} more lines, in the file")
            return
    print(f"{obj} is not built by a SQL module.")


def show_py(filename, func, lines=None):
    """Print one function from a production file in src/."""
    rows = (ROOT / "src" / filename).read_text(encoding="utf-8").splitlines()
    start = next(i for i, r in enumerate(rows) if r.startswith(f"def {func}("))
    end = start + 1
    while end < len(rows) and not rows[end].startswith(("def ", "class ", "@")):
        end += 1
    block = rows[start:end]
    while block and not block[-1].strip():
        block.pop()
    shown = block if lines is None else block[:lines]
    print(f"--- src/{filename} - def {func} - {len(block)} lines " + "-" * 16)
    print("\n".join(shown))
    if lines is not None and len(block) > lines:
        print(f"... {len(block) - lines} more lines, in the file")

print(f"Warehouse : {DB.name}")
print(f"Size      : {DB.stat().st_size / 1_048_576:.0f} MB")

Warehouse : thelook_analytics.duckdb
Size      : 270 MB


In [2]:
# What the pipeline produced, by layer.
display(q("""
    SELECT table_schema AS layer, COUNT(*) AS objects
    FROM information_schema.tables
    WHERE table_schema IN ('raw','stg','core','mart','qa')
    GROUP BY 1
    ORDER BY CASE table_schema WHEN 'raw' THEN 1 WHEN 'stg' THEN 2
                               WHEN 'core' THEN 3 WHEN 'mart' THEN 4 ELSE 5 END
"""))

,layer,objects
0,raw,10
1,stg,9
2,core,14
3,mart,27
4,qa,6


---

## Part 2 - The sources

**What it does.** Three independent inputs feed this project. The seven TheLook
CSVs are the transactional source. Two committed seed files carry reference data
the source does not contain. A regional plan submission file is a genuine second
source system, arriving in local currency with the defects a real planning cycle
produces.

| | |
|---|---|
| **Built by** | `src/pipeline.py` (ingestion), `src/build_plan_source.py` (plan file) |
| **Written to** | the `raw` schema in `artifacts/thelook_analytics.duckdb` |


In [3]:
display(q("""
    SELECT table_name, estimated_size AS rows
    FROM duckdb_tables() WHERE schema_name = 'raw'
    ORDER BY rows DESC
"""))

,table_name,rows
0,events,2420661
1,inventory_events,488146
2,order_items,180862
3,orders,124814
4,users,100000
5,products,29120
6,plan_submissions,292
7,country_master,16
8,distribution_centers,10
9,fx_budget_rates,10


### Why the seeds are committed rather than derived

`seeds/country_master.csv` maps every country spelling the source emits onto a
canonical market. `seeds/fx_budget_rates.csv` holds the budget FX rate fixed at
the planning-cycle open. Neither can be inferred from transactional data - they
are decisions, so they are version-controlled alongside the code.


In [4]:
print("The alias rows are the ones that matter:")
display(q("""
    SELECT source_country_value, canonical_country, region, is_alias, alias_reason
    FROM raw.country_master
    WHERE is_alias = 1 OR canonical_country IN ('Brazil','Spain','Germany')
    ORDER BY canonical_country, is_alias DESC
"""))

The alias rows are the ones that matter:


,source_country_value,canonical_country,region,is_alias,alias_reason
0,Brasil,Brazil,AMER,1,Portuguese-language spelling emitted by the source system
1,Deutschland,Germany,EMEA,1,Local-language spelling of Germany found in source users.csv
2,Germany,Germany,EMEA,0,NaN
3,España,Spain,EMEA,1,Local-language spelling of Spain found in source users.csv
4,Spain,Spain,EMEA,0,NaN


> **Conclusion.** TheLook's own `users.csv` carries **15 country spellings for 13 markets** - `Brasil` and `Brazil`, `Espana` and `Spain`, `Deutschland` and `Germany`. This is a master-data defect present in the source, not one introduced here, and it is the reason a country dimension exists at all.


---

## Part 3 - The source quality gate

**What it does.** Nine checks that need nothing but the `raw` schema, run before a
single model is built. If a source is broken there is no reason to compute
twenty-six marts on it. A failure here stops the build before any downstream
object exists and before anything is exported.

Two of the checks catch a **new** value rather than a known bad one, which is the
difference between a test that ages well and one that does not:

- `country_master_covers_source` - every spelling in `users.csv` must map to a market
- `fx_rates_cover_plan_currencies` - every plan currency must have a budget rate

| | |
|---|---|
| **Built by** | `sql/duckdb/00_raw_quality.sql`, enforced by `assert_no_blocking_failures()` in `src/pipeline.py` |
| **Written to** | `qa.raw_test_results`, exported to `outputs/qa/raw_test_results.csv` |


In [5]:
display(q("""
    SELECT check_name, severity, failure_count, denominator, status, notes
    FROM qa.raw_test_results
    ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'WARN' THEN 2 ELSE 3 END, check_name
"""))

,check_name,severity,failure_count,denominator,status,notes
0,events_user_id_missing,INFO,1124527,2420661,WARN,Event user_id is sparse; session customer_id uses the one nonnull ...
1,country_master_covers_source,CRITICAL,0,15,PASS,Every country spelling in users.csv must map to a canonical market
2,events_primary_key_unique,CRITICAL,0,2420661,PASS,events.id must be unique
3,events_session_id_present,CRITICAL,0,2420661,PASS,Every event must have a nonblank session_id before sessionization
4,fx_rates_cover_plan_currencies,CRITICAL,0,10,PASS,Every currency in the plan file must have a budget FX rate
5,order_items_primary_key_unique,CRITICAL,0,180862,PASS,order_items.id must be unique
6,orders_primary_key_unique,CRITICAL,0,124814,PASS,orders.order_id must be unique
7,products_primary_key_unique,CRITICAL,0,29120,PASS,products.id must be unique
8,users_primary_key_unique,CRITICAL,0,100000,PASS,users.id must be unique


> **Conclusion.** Nine checks, zero blocking failures. The one WARN is an INFO-severity observation that event `user_id` is sparse - a known source condition, reported every build and deliberately non-blocking. **The gate is enforced, not just reported**: the pipeline raises `QualityGateFailed` and stops before the staging layer if any blocking check fails.


---

## Part 4 - Staging

**What it does.** One view per source table. Types are cast, columns renamed to a
consistent convention, and nothing else. No joins, no filtering, no business logic -
so any warehouse row can be traced back to the source row it came from.

| | |
|---|---|
| **Built by** | `sql/duckdb/01_staging.sql` |
| **Written to** | the `stg` schema (views, not tables - they cost nothing to keep current) |


In [6]:
# The production SQL, read from the file the pipeline runs.
show_sql("stg.order_items")

--- sql/duckdb/01_staging.sql - 14 lines ----------------------
CREATE OR REPLACE VIEW stg.order_items AS
SELECT
    CAST(id AS BIGINT) AS order_item_id,
    CAST(order_id AS BIGINT) AS order_id,
    CAST(user_id AS BIGINT) AS user_id,
    CAST(product_id AS BIGINT) AS product_id,
    CAST(inventory_item_id AS BIGINT) AS inventory_item_id,
    TRIM(CAST(status AS VARCHAR)) AS item_status,
    CAST(created_at AS TIMESTAMP) AS created_at,
    CAST(shipped_at AS TIMESTAMP) AS shipped_at,
    CAST(delivered_at AS TIMESTAMP) AS delivered_at,
    CAST(returned_at AS TIMESTAMP) AS returned_at,
    TRY_CAST(sale_price AS DOUBLE) AS sale_price
FROM raw.order_items;


In [7]:
display(q("""
    SELECT table_name, table_type
    FROM information_schema.tables WHERE table_schema = 'stg'
    ORDER BY table_name
"""))

,table_name,table_type
0,distribution_centers,VIEW
1,events,VIEW
2,inventory_events,VIEW
3,order_items,VIEW
4,orders,VIEW
5,plan_submissions,VIEW
6,products,VIEW
7,sessions,VIEW
8,users,VIEW


> **Conclusion.** Staging is deliberately boring. Keeping it a one-to-one mirror of the source is what makes the reconciliation in Part 13 possible - row counts and values must match the source exactly at this layer, so any later discrepancy is attributable to a specific modelling step.


---

## Part 5 - Core - conformed dimensions

**What it does.** The dimensions every fact joins to. This is where the country
master is applied, so `Brasil` and `Brazil` become one market everywhere downstream.

One modelling decision is worth stating. `dim_country` is a **mapping table** - one
row per observed source spelling, so `canonical_country` repeats inside it. It
therefore cannot serve as the "one" side of a BI relationship. `dim_market` exists
separately with one row per market, and `dim_month` exists for the same reason
because `dim_date` is at day grain.

| | |
|---|---|
| **Built by** | `sql/duckdb/02_core_dimensions.sql` |
| **Written to** | the `core` schema; `dim_market`, `dim_month` and `dim_country` are exported to `data/processed/power_bi/` |


In [8]:
# How the country master is applied. This is the join that ends the Brasil/Brazil problem.
show_sql("core.dim_country")

--- sql/duckdb/02_core_dimensions.sql - 10 lines ----------------------
CREATE OR REPLACE TABLE core.dim_country AS
SELECT
    ROW_NUMBER() OVER (ORDER BY canonical_country, source_country_value)::INTEGER AS country_key,
    source_country_value,
    canonical_country,
    region,
    sub_region,
    CAST(is_alias AS INTEGER) AS is_alias,
    alias_reason
FROM raw.country_master;


In [9]:
print("dim_country - a mapping table, canonical_country repeats:")
display(q("SELECT COUNT(*) AS rows, COUNT(DISTINCT canonical_country) AS markets FROM core.dim_country"))

print("\ndim_market - one row per market, safe as a BI dimension:")
display(q("SELECT COUNT(*) AS rows, COUNT(DISTINCT canonical_country) AS markets FROM core.dim_market"))

dim_country - a mapping table, canonical_country repeats:


,rows,markets
0,16,14



dim_market - one row per market, safe as a BI dimension:


,rows,markets
0,13,13


> **Conclusion.** Pointing a BI relationship at `dim_country` would create a many-to-many join and silently double every market total. Splitting the mapping table from the dimension is not pedantry - it is the difference between a report that is right and one that looks right.


---

## Part 6 - Core - facts

**What it does.** The transactional grain. `fact_order_item` is one row per line
item, enriched with the product, customer and date keys, and carrying the derived
measures the marts aggregate: gross sales, net sales, net profit and the status
flags that define what "sold" means.

| | |
|---|---|
| **Built by** | `sql/duckdb/03_core_facts.sql` |
| **Written to** | the `core` schema; `fact_order_item` is exported to `data/processed/power_bi/` |


In [10]:
# Where gross, net and profit are defined. Every mart number traces back to these CASE expressions.
show_sql("core.fact_order_item", lines=40)

--- sql/duckdb/03_core_facts.sql - 40 lines ----------------------
CREATE OR REPLACE TABLE core.fact_order_item AS
SELECT
    i.order_item_id,
    i.order_id,
    i.user_id AS customer_id,
    i.product_id,
    i.inventory_item_id,
    p.distribution_center_id,
    c.acquisition_source AS acquisition_source,
    i.item_status,
    i.created_at AS order_item_created_at,
    i.shipped_at,
    i.delivered_at,
    i.returned_at,
    CAST(strftime(i.created_at, '%Y%m%d') AS INTEGER) AS order_date_key,
    i.sale_price,
    p.unit_cost,
    i.sale_price - p.unit_cost AS gross_margin_value,
    CASE WHEN i.sale_price > 0 THEN (i.sale_price - p.unit_cost) / i.sale_price END AS gross_margin_rate,
    i.sale_price AS gross_sales_value,
    CASE WHEN i.item_status = 'Cancelled' THEN i.sale_price ELSE 0 END AS cancelled_value,
    CASE WHEN i.item_status = 'Returned' THEN i.sale_price ELSE 0 END AS returned_value,
    CASE WHEN i.item_status NOT IN ('Cancelled', 'Returned') THEN i.sale_price ELSE 

In [11]:
display(q("""
    SELECT COUNT(*) AS line_items,
           COUNT(DISTINCT order_id) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           ROUND(SUM(gross_sales_value), 0) AS gross_sales,
           ROUND(SUM(net_sales_value), 0) AS net_sales,
           ROUND(SUM(net_profit_value), 0) AS net_profit
    FROM core.fact_order_item
"""))

,line_items,orders,customers,gross_sales,net_sales,net_profit
0,180862,124814,80095,10796336.0,8096531.0,4201367.0


In [12]:
# Gross minus net is not noise - it is cancelled and returned value.
display(q("""
    SELECT item_status, COUNT(*) AS line_items,
           ROUND(SUM(gross_sales_value), 0) AS gross_value
    FROM core.fact_order_item GROUP BY 1 ORDER BY line_items DESC
"""))

,item_status,line_items,gross_value
0,Shipped,54456,3226035.0
1,Complete,45422,2729363.0
2,Processing,35871,2141133.0
3,Cancelled,27072,1623638.0
4,Returned,18041,1076166.0


> **Conclusion.** **One line item in four never completes** - 27,072 cancelled and 18,041 returned out of 180,862. That 25% gap between gross and net is the single largest commercial fact in the dataset, and it is why every mart reports net rather than gross.


---

## Part 7 - Marts - acquisition and funnel

**What it does.** Session-level behaviour rebuilt from 2.4 million raw events:
channel quality, the session funnel from view to cart to purchase, and cart
abandonment by segment.

| | |
|---|---|
| **Built by** | `sql/duckdb/04_mart_acquisition_funnel.sql` |
| **Written to** | `mart.channel_quality`, `mart.funnel_monthly_channel`, `mart.cart_abandonment_segments`, `mart.acquisition_source_value` |


In [13]:
display(q("""
    SELECT traffic_source, sessions, purchase_sessions,
           ROUND(session_conversion_rate, 4) AS conversion_rate,
           ROUND(cart_abandonment_rate, 4) AS abandonment_rate
    FROM mart.channel_quality ORDER BY sessions DESC
"""))

,traffic_source,sessions,purchase_sessions,conversion_rate,abandonment_rate
0,Email,306377,81364.0,0.2656,0.5800
1,Adwords,203690,54072.0,0.2655,0.5797
2,YouTube,68524,18172.0,0.2652,0.5811
3,Facebook,68518,18343.0,0.2677,0.5788
4,Organic,33753,8911.0,0.2640,0.5822


![Session funnel by acquisition channel](../artifacts/figures/funnel_by_channel.png)

*Session funnel by acquisition channel*


> **Conclusion.** Cart-to-purchase conversion is **uniform across every channel**. Channels differ in the volume they bring, not in the quality of the traffic, so the lever is acquisition spend allocation rather than channel-specific funnel optimisation.


---

## Part 8 - Marts - revenue, product and customer

**What it does.** The descriptive core: monthly sales and profit, category and brand
performance, geography, cohort retention and the customer 360 table.

| | |
|---|---|
| **Built by** | `sql/duckdb/05_mart_revenue_customer.sql` |
| **Written to** | `mart.sales_monthly`, `mart.product_performance_category`, `mart.product_performance_brand`, `mart.geography_performance_country`, `mart.cohort_retention`, `mart.customer_360` |


In [14]:
display(q("""
    SELECT month_start, ROUND(net_sales_value, 0) AS net_sales,
           ROUND(net_profit_value, 0) AS net_profit,
           ROUND(net_margin_rate, 4) AS margin_rate, orders, customers
    FROM mart.sales_monthly WHERE is_complete_month = 1
    ORDER BY month_start DESC LIMIT 12
"""))

,month_start,net_sales,net_profit,margin_rate,orders,customers
0,2024-04-01,550158.0,285251.0,0.5185,8703,7656
1,2024-03-01,441334.0,229098.0,0.5191,7122,6568
2,2024-02-01,381973.0,198564.0,0.5198,5813,5500
3,2024-01-01,348738.0,180966.0,0.5189,5539,5307
4,2023-12-01,326302.0,168955.0,0.5178,5221,5013
5,2023-11-01,298468.0,155330.0,0.5204,4525,4363
6,2023-10-01,279982.0,145193.0,0.5186,4447,4320
7,2023-09-01,261139.0,135291.0,0.5181,4091,3986
8,2023-08-01,246130.0,126847.0,0.5154,3947,3837
9,2023-07-01,232778.0,121048.0,0.5200,3653,3558


![Monthly net sales and net profit](../artifacts/figures/monthly_sales_profit.png)

*Monthly net sales and net profit*


![Category value and margin - where the money and the margin are](../artifacts/figures/category_sales_margin.png)

*Category value and margin - where the money and the margin are*


In [15]:
print("Geography, after the country master is applied:")
display(q("""
    SELECT country, region, ROUND(net_sales_value, 0) AS net_sales,
           customers, source_spellings
    FROM mart.geography_performance_country ORDER BY net_sales DESC
"""))

Geography, after the country master is applied:


,country,region,net_sales,customers,source_spellings
0,China,APAC,2744128.0,27184,China
1,United States,AMER,1803753.0,17803,United States
2,Brazil,AMER,1170131.0,11706,Brasil
3,South Korea,APAC,425596.0,4258,South Korea
4,France,EMEA,382267.0,3736,France
5,United Kingdom,EMEA,381532.0,3779,United Kingdom
6,Germany,EMEA,353573.0,3427,Germany | Deutschland
7,Spain,EMEA,335188.0,3270,España | Spain
8,Japan,APAC,199997.0,1972,Japan
9,Australia,APAC,179206.0,1770,Australia


![Cohort retention by acquisition month](../artifacts/figures/cohort_retention_heatmap.png)

*Cohort retention by acquisition month*


> **Conclusion.** Net margin sits at a flat **51.8%** every month across five years, which is a property of how TheLook generates its data rather than a commercial finding - worth stating plainly rather than presenting as a margin achievement. The `source_spellings` column is the audit trail: it shows which raw spellings were folded into each market.


---

## Part 9 - Marts - operations and inventory

**What it does.** Delivery lead time decomposed into warehouse processing and carrier
transit, return and cancellation rates over time, and inventory sell-through and
ageing by category and distribution centre.

| | |
|---|---|
| **Built by** | `sql/duckdb/06_mart_operations_inventory.sql` |
| **Written to** | `mart.operations_monthly`, `mart.delivery_performance_dc`, `mart.inventory_performance`, `mart.return_risk_segments` |


In [16]:
display(q("""
    SELECT distribution_center_name,
           ROUND(avg_ship_lead_days, 2) AS warehouse_days,
           ROUND(avg_delivery_lead_days, 2) AS carrier_days,
           ROUND(avg_end_to_end_lead_days, 2) AS total_days
    FROM mart.delivery_performance_dc ORDER BY total_days DESC LIMIT 10
"""))

,distribution_center_name,warehouse_days,carrier_days,total_days
0,New Orleans LA,1.47,2.51,3.54
1,Memphis TN,1.47,2.51,3.51
2,Mobile AL,1.47,2.52,3.50
3,Los Angeles CA,1.46,2.50,3.49
4,Savannah GA,1.45,2.49,3.48
5,Chicago IL,1.47,2.48,3.47
6,Port Authority of New York/New Jersey NY/NJ,1.46,2.50,3.47
7,Houston TX,1.45,2.46,3.45
8,Philadelphia PA,1.44,2.48,3.45
9,Charleston SC,1.46,2.46,3.44


![Delivery lead time by distribution centre](../artifacts/figures/delivery_lead_time_dc.png)

*Delivery lead time by distribution centre*


> **Conclusion.** Lead time is **identical across all ten distribution centres** - 1.5 days in the warehouse and 2.5 days in transit. There is no underperforming site to fix; the only lever on delivery speed is the carrier contract, which is a procurement decision rather than an operations one.


---

## Part 10 - Warehouse quality gates

**What it does.** The checks that compare one layer against another - staged rows
versus modelled rows, referential integrity, timestamp ordering. They cannot run any
earlier than this because they need the warehouse to exist. The source-level checks
already ran in Part 3.

| | |
|---|---|
| **Built by** | `sql/duckdb/07_quality_and_snapshots.sql` |
| **Written to** | `qa.test_results` and `qa.metric_reconciliation`, exported to `outputs/qa/` |


In [17]:
display(q("""
    SELECT check_name, layer, severity, failure_count, denominator, status
    FROM qa.test_results
    ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'WARN' THEN 2 ELSE 3 END, check_name
"""))

,check_name,layer,severity,failure_count,denominator,status
0,partial_latest_month,mart,INFO,1.0,1,WARN
1,shipping_not_before_item_created,core,HIGH,35440.0,180862,WARN
2,delivery_not_before_shipping,core,HIGH,0.0,180862,PASS
3,derived_sessions_primary_key_unique,stg,CRITICAL,0.0,680862,PASS
4,nonnegative_sale_price,stg,HIGH,0.0,180862,PASS
5,order_item_count_reconciles,core,CRITICAL,0.0,180862,PASS
6,order_items_have_order,stg,CRITICAL,0.0,180862,PASS
7,order_items_have_product,stg,CRITICAL,0.0,180862,PASS
8,orders_have_customer,stg,CRITICAL,0.0,124814,PASS
9,product_cost_not_above_retail,stg,HIGH,0.0,29120,PASS


> **Conclusion.** Fourteen checks, zero blocking failures. The two WARNs are known source conditions - sparse event user IDs and a small number of invalid timestamp sequences - which are reported on every build and excluded from the lead-time calculations rather than silently included.


---

## Part 11 - The second source - regional plan submissions

**What it does.** TheLook ships no budget data, so a planning file is constructed from
a transparent, deterministic rule and written to `data/planning/` as a committed
source artifact. It is then treated exactly like any other source system - ingested,
gated, staged and validated with no privileged knowledge of how it was produced.

```
plan(country, month) = actual(same month last year)
                     x trailing_growth(country, planning cycle)
                     x ambition(country)
                     x seasonal_uplift(month)
```

`trailing_growth` is the country's own CY(n-1)/CY(n-2) rate - the last complete year
a planner could have seen when writing the plan. Planning off a trailing rate rather
than off hindsight is what makes the resulting variance realistic instead of
trivially positive.

**Defects deliberately present**, because a validation suite that only ever passes
proves nothing: local-language country names, one market submitting two months late,
four blank cells, one negative target, and trailing whitespace on a country key.

| | |
|---|---|
| **Built by** | `src/build_plan_source.py` |
| **Written to** | `data/planning/regional_plan_submissions.csv`, ingested to `raw.plan_submissions` |


In [18]:
print("The source file, as submitted - local currency, no USD:")
display(q("SELECT * FROM raw.plan_submissions LIMIT 12"))

print("\nRows carrying a defect:")
display(q("""
    SELECT submission_batch, submitted_country, plan_month, currency_code,
           plan_net_sales_local,
           CASE WHEN plan_net_sales_local IS NULL THEN 'blank target'
                WHEN plan_net_sales_local < 0 THEN 'negative target'
                WHEN submitted_country <> TRIM(submitted_country) THEN 'trailing whitespace'
                ELSE 'non-English country name' END AS defect
    FROM raw.plan_submissions
    WHERE plan_net_sales_local IS NULL OR plan_net_sales_local < 0
       OR submitted_country <> TRIM(submitted_country)
       OR submitted_country NOT IN (SELECT canonical_country FROM raw.country_master)
    ORDER BY defect, submitted_country, plan_month LIMIT 12
"""))

The source file, as submitted - local currency, no USD:


,submission_batch,submitted_country,plan_month,currency_code,plan_net_sales_local,plan_units
0,FY2023-PLAN-V1,Australia,2023-01-01,AUD,5000.00,52
1,FY2023-PLAN-V1,Australia,2023-02-01,AUD,1969.70,33
2,FY2023-PLAN-V1,Australia,2023-03-01,AUD,3787.88,63
3,FY2023-PLAN-V1,Australia,2023-04-01,AUD,7272.73,74
4,FY2023-PLAN-V1,Australia,2023-05-01,AUD,6212.12,76
5,FY2023-PLAN-V1,Australia,2023-06-01,AUD,6515.15,73
6,FY2023-PLAN-V1,Australia,2023-07-01,AUD,7575.76,77
7,FY2023-PLAN-V1,Australia,2023-08-01,AUD,6969.70,74
8,FY2023-PLAN-V1,Australia,2023-09-01,AUD,8787.88,85
9,FY2023-PLAN-V1,Australia,2023-10-01,AUD,7121.21,88



Rows carrying a defect:


,submission_batch,submitted_country,plan_month,currency_code,plan_net_sales_local,defect
0,FY2024-PLAN-V1,Australia,2024-02-01,AUD,NaN,blank target
1,FY2023-PLAN-V1,France,2023-08-01,EUR,NaN,blank target
2,FY2023-PLAN-V1,Japan,2023-11-01,JPY,NaN,blank target
3,FY2024-PLAN-V1,United Kingdom,2024-09-01,GBP,NaN,blank target
4,FY2023-PLAN-V1,Belgium,2023-05-01,EUR,-2685.19,negative target
5,FY2023-PLAN-V1,Brasil,2023-01-01,BRL,183500.00,non-English country name
6,FY2023-PLAN-V1,Brasil,2023-02-01,BRL,129000.00,non-English country name
7,FY2023-PLAN-V1,Brasil,2023-03-01,BRL,153500.00,non-English country name
8,FY2023-PLAN-V1,Brasil,2023-04-01,BRL,119500.00,non-English country name
9,FY2023-PLAN-V1,Brasil,2023-05-01,BRL,153000.00,non-English country name


> **Conclusion.** The plan file is **constructed, and the construction rule is published** in the module docstring and in `docs/commercial_analytics.md`. The alternative - inventing a budget with no stated rule - would make every variance number in this project unfalsifiable.


---

## Part 12 - The commercial layer

**What it does.** Harmonises the submission against the country master, converts local
currency to USD at the fixed budget rate, consolidates to a conformed country-month
grain, and builds the plan-versus-actual and price-volume-mix layers on top.

The staging view keeps every submitted row and adds judgement as **flag columns**
rather than filtering. `core.fact_plan` then takes only the rows that pass. That
separation is what makes a rejected row traceable back to the submission it came from.

| | |
|---|---|
| **Built by** | `sql/duckdb/08_commercial_planning.sql` |
| **Written to** | `stg.plan_submissions`, `core.fact_plan`, `mart.plan_vs_actual_monthly`, `mart.price_volume_mix`, `mart.commercial_scorecard` |


In [19]:
# Harmonisation, FX conversion and the flag columns - all in one view.
show_sql("stg.plan_submissions", lines=45)

--- sql/duckdb/08_commercial_planning.sql - 45 lines ----------------------
CREATE OR REPLACE VIEW stg.plan_submissions AS
WITH trimmed AS (
    SELECT
        submission_batch,
        submitted_country,
        TRIM(submitted_country) AS submitted_country_trimmed,
        CAST(plan_month AS DATE) AS plan_month,
        UPPER(TRIM(currency_code)) AS currency_code,
        CAST(plan_net_sales_local AS DOUBLE) AS plan_net_sales_local,
        CAST(plan_units AS BIGINT) AS plan_units
    FROM raw.plan_submissions
),
mapped AS (
    SELECT
        t.*,
        c.canonical_country,
        c.region,
        c.sub_region,
        f.budget_rate_to_usd
    FROM trimmed t
    LEFT JOIN (
        SELECT DISTINCT source_country_value, canonical_country, region, sub_region
        FROM core.dim_country
    ) c ON t.submitted_country_trimmed = c.source_country_value
    LEFT JOIN raw.fx_budget_rates f ON t.currency_code = f.currency_code
)
SELECT
    submission_batch,
    submitted_country,
    su

In [20]:
print("One row traced through all four layers:")
print("\nRAW - exactly as submitted")
display(q("""SELECT * FROM raw.plan_submissions
             WHERE submitted_country = 'Deutschland' AND plan_month = '2023-01-01'"""))

print("STG - harmonised, FX-converted, flagged")
display(q("""SELECT submitted_country, canonical_country, region, currency_code,
                    budget_rate_to_usd, plan_net_sales_local, plan_net_sales_usd,
                    flag_unmapped_country, flag_negative_target
             FROM stg.plan_submissions
             WHERE submitted_country = 'Deutschland' AND plan_month = DATE '2023-01-01'"""))

print("CORE - accepted rows only, on the conformed grain")
display(q("""SELECT canonical_country, region, plan_month, plan_net_sales_usd, plan_units
             FROM core.fact_plan
             WHERE canonical_country = 'Germany' AND plan_month = DATE '2023-01-01'"""))

One row traced through all four layers:

RAW - exactly as submitted


,submission_batch,submitted_country,plan_month,currency_code,plan_net_sales_local,plan_units
0,FY2023-PLAN-V1,Deutschland,2023-01-01,EUR,9444.44,150


STG - harmonised, FX-converted, flagged


,submitted_country,canonical_country,region,currency_code,budget_rate_to_usd,plan_net_sales_local,plan_net_sales_usd,flag_unmapped_country,flag_negative_target
0,Deutschland,Germany,EMEA,EUR,1.08,9444.44,10199.9952,0,0


CORE - accepted rows only, on the conformed grain


,canonical_country,region,plan_month,plan_net_sales_usd,plan_units
0,Germany,EMEA,2023-01-01,10199.9952,150.0


### Plan versus actual

A full outer join, so three outcomes are visible rather than two: months with both,
months with an actual and no plan, and months with a plan and no actual yet.


In [21]:
# The full outer join, and the COALESCE that rebuilds the key from whichever side exists.
show_sql("mart.plan_vs_actual_monthly", lines=40)

--- sql/duckdb/08_commercial_planning.sql - 30 lines ----------------------
CREATE OR REPLACE TABLE mart.plan_vs_actual_monthly AS
SELECT
    COALESCE(a.canonical_country, p.canonical_country) AS canonical_country,
    COALESCE(a.region, p.region) AS region,
    COALESCE(a.month_start, p.plan_month) AS month_start,
    a.net_sales_value AS actual_net_sales_usd,
    a.net_units AS actual_units,
    p.plan_net_sales_usd,
    p.plan_units,
    a.net_sales_value - p.plan_net_sales_usd AS variance_usd,
    CASE WHEN p.plan_net_sales_usd > 0
         THEN (a.net_sales_value - p.plan_net_sales_usd) / p.plan_net_sales_usd END AS variance_rate,
    CASE WHEN p.plan_net_sales_usd > 0
         THEN a.net_sales_value / p.plan_net_sales_usd END AS achievement_rate,
    COALESCE(a.is_complete_month, 0) AS is_complete_month,
    CASE WHEN COALESCE(a.month_start, p.plan_month)
              BETWEEN (SELECT MIN(plan_month) FROM core.fact_plan)
                  AND (SELECT MAX(plan_month) FROM core.fac

In [22]:
display(q("""
    SELECT plan_status, COUNT(*) AS months,
           ROUND(SUM(actual_net_sales_usd), 0) AS actual_usd,
           ROUND(SUM(plan_net_sales_usd), 0) AS plan_usd
    FROM mart.plan_vs_actual_monthly GROUP BY 1 ORDER BY months DESC
"""))

,plan_status,months,actual_usd,plan_usd
0,Actual with no plan,573,3181018.0,NaN
1,On or above plan,111,3014105.0,2457900.0
2,Plan with no actual yet,90,NaN,3475800.0
3,Below plan,64,1334208.0,1565800.0
4,Within 5% of plan,22,567200.0,581800.0


In [23]:
# Like-for-like achievement: complete months where both a plan and an actual exist.
display(q("""
    SELECT canonical_country,
           ROUND(SUM(actual_net_sales_usd), 0) AS actual_usd,
           ROUND(SUM(plan_net_sales_usd), 0) AS plan_usd,
           ROUND(SUM(actual_net_sales_usd) / SUM(plan_net_sales_usd), 3) AS achievement
    FROM mart.plan_vs_actual_monthly
    WHERE is_complete_month = 1 AND plan_net_sales_usd IS NOT NULL
      AND actual_net_sales_usd IS NOT NULL
    GROUP BY 1 ORDER BY achievement
"""))

,canonical_country,actual_usd,plan_usd,achievement
0,Poland,8357.0,11600.0,0.720
1,Colombia,85.0,100.0,0.850
2,China,1527852.0,1506800.0,1.014
3,Japan,102493.0,99200.0,1.033
4,South Korea,244565.0,235900.0,1.037
5,United States,1012384.0,961500.0,1.053
6,Belgium,52944.0,50100.0,1.057
7,Brazil,648034.0,588200.0,1.102
8,Australia,88594.0,78400.0,1.130
9,Germany,197974.0,174300.0,1.136


![Plan achievement by market](../artifacts/figures/plan_achievement_by_market.png)

*Plan achievement by market*


> **Conclusion.** Group achievement is **106.2% on a like-for-like basis**. Two markets are below plan: Poland at 72%, because it submitted two months late and those months carry revenue with no target against them, and Colombia at 85% on a base so small that the plan is barely meaningful. Eleven markets are between 101% and 117%.


### The price-volume-mix bridge

Revenue is three numbers multiplied together - market units, this category's share of
them, and its average selling price. Changing one at a time and measuring each step
decomposes any revenue movement into exactly three causes.

```
volume = (Q1 - Q0) x s0 x P0      the market grew, we were carried with it
mix    =  Q1 x (s1 - s0) x P0     we took a different share of it
price  =  q1 x (P1 - P0)          we sold at a different price
```

The sequence is fixed and not arbitrary: volume is measured first and therefore at the
prior share and price, mix second at the new market size but the old price, price last
at current volume and share. That ordering is what makes the three effects sum to the
variance exactly, with no residual bucket.


In [24]:
# The three effects, computed in SQL. The sided CTE is where the ordering is enforced.
show_sql("mart.price_volume_mix", lines=45)

--- sql/duckdb/08_commercial_planning.sql - 93 lines ----------------------
CREATE OR REPLACE TABLE mart.price_volume_mix AS
WITH comparisons(comparison_label, comparison_order,
                 prior_from, prior_to, current_from, current_to) AS (
    VALUES
        ('CY2023 vs CY2022', 1,
         DATE '2022-01-01', DATE '2022-12-31', DATE '2023-01-01', DATE '2023-12-31'),
        ('YTD Jan-Apr 2024 vs 2023', 2,
         DATE '2023-01-01', DATE '2023-04-30', DATE '2024-01-01', DATE '2024-04-30'),
        ('Apr 2024 vs Apr 2023', 3,
         DATE '2023-04-01', DATE '2023-04-30', DATE '2024-04-01', DATE '2024-04-30')
),
items AS (
    SELECT
        f.order_item_created_at::DATE AS activity_date,
        p.department,
        p.category,
        f.net_sales_value
    FROM core.fact_order_item f
    JOIN core.dim_product p ON f.product_id = p.product_id
    WHERE f.item_status NOT IN ('Cancelled', 'Returned')
),
sided AS (
    SELECT
        c.comparison_label,
        c.comparison_order

In [25]:
display(q("""
    SELECT comparison_label,
           ROUND(SUM(revenue_prior), 0) AS revenue_prior,
           ROUND(SUM(revenue_current), 0) AS revenue_current,
           ROUND(SUM(total_variance), 0) AS total_variance,
           ROUND(SUM(volume_effect), 0) AS volume,
           ROUND(SUM(mix_effect), 0) AS mix,
           ROUND(SUM(price_effect), 0) AS price,
           ROUND(ABS(SUM(volume_effect) + SUM(mix_effect) + SUM(price_effect)
                     - SUM(total_variance)), 6) AS residual
    FROM mart.price_volume_mix GROUP BY 1, comparison_order ORDER BY comparison_order
"""))

,comparison_label,revenue_prior,revenue_current,total_variance,volume,mix,price,residual
0,CY2023 vs CY2022,1583799.0,2793942.0,1210144.0,1229590.0,-9499.0,-9948.0,0.0
1,YTD Jan-Apr 2024 vs 2023,734018.0,1722202.0,988185.0,972255.0,-6177.0,22107.0,0.0
2,Apr 2024 vs Apr 2023,193150.0,550158.0,357008.0,349337.0,-5773.0,13444.0,0.0


![Price-volume-mix bridge](../artifacts/figures/price_volume_mix_bridge.png)

*Price-volume-mix bridge*


> **Conclusion.** Growth in CY2023 was **almost entirely volume** - 1.23 million of a 1.21 million variance, with mix and price both slightly negative. The company grew by selling more, while quietly discounting. By April 2024 the price effect had turned positive at +13,444 and average selling price had risen from 59.52 to 60.36, which is a different and better quality of growth. The residual is zero on all three comparisons.


---

## Part 13 - Commercial quality gates and reconciliation

**What it does.** Twelve checks on the commercial layer, and an end-to-end
reconciliation statement that can be handed to a reviewer as it is.

The design decision matters more than the checks. **Structural defects block the
build** - an unmapped country, a broken grain, an FX conversion that does not
reproduce, a consolidation that does not tie, a bridge whose effects do not sum.
Each of those makes the consolidated number *wrong*, and publishing a wrong number is
worse than publishing nothing. **Submission defects quarantine and warn** - blank
cells, a sign error, a late region. Those make the plan *incomplete* rather than
wrong; the month-end review still has to happen, and the defect goes back to the
submitting market.

| | |
|---|---|
| **Built by** | `sql/duckdb/09_quality_commercial.sql` |
| **Written to** | `qa.plan_validation`, `qa.plan_rejects`, `qa.reconciliation_statement`, exported to `outputs/qa/` |


In [26]:
# The twelve checks, as written. Severity is set per check, and severity is what decides whether a build stops.
show_sql("qa.plan_validation", lines=45)

--- sql/duckdb/09_quality_commercial.sql - 117 lines ----------------------
CREATE OR REPLACE TABLE qa.plan_validation AS
WITH plan_month_bounds AS (
    SELECT canonical_country, MIN(plan_month) AS first_month, MAX(plan_month) AS last_month
    FROM core.fact_plan
    GROUP BY 1
),
expected_months AS (
    SELECT b.canonical_country, CAST(m.value AS DATE) AS plan_month
    FROM plan_month_bounds b,
         generate_series(b.first_month, b.last_month, INTERVAL 1 MONTH) AS m(value)
),
month_gaps AS (
    SELECT COUNT(*) AS gap_count
    FROM expected_months e
    LEFT JOIN core.fact_plan p
      ON e.canonical_country = p.canonical_country AND e.plan_month = p.plan_month
    WHERE p.canonical_country IS NULL
),
pvm_residual AS (
    SELECT COUNT(*) AS broken_comparisons
    FROM (
        SELECT comparison_label,
               ABS(SUM(volume_effect) + SUM(mix_effect) + SUM(price_effect)
                   - SUM(total_variance)) AS residual
        FROM mart.price_volume_mix
        GR

In [27]:
print("Validation")
display(q("""SELECT check_name, severity, failure_count, denominator, status
             FROM qa.plan_validation
             ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'WARN' THEN 2 ELSE 3 END,
                      check_name"""))

Validation


,check_name,severity,failure_count,denominator,status
0,plan_country_key_clean,MEDIUM,7,292,WARN
1,plan_month_coverage_contiguous,HIGH,24,287,WARN
2,plan_target_not_blank,HIGH,4,292,WARN
3,plan_target_not_negative,HIGH,1,292,WARN
4,actuals_tie_to_fact_order_item,CRITICAL,0,1,PASS
5,plan_consolidation_ties_to_submissions,CRITICAL,0,1,PASS
6,plan_country_resolves_to_master,CRITICAL,0,292,PASS
7,plan_currency_has_budget_rate,CRITICAL,0,292,PASS
8,plan_fx_conversion_reconciles,CRITICAL,0,292,PASS
9,plan_grain_unique_country_month,CRITICAL,0,287,PASS


In [28]:
print("Quarantined rows - the list that goes back to the submitting market")
display(q("SELECT * FROM qa.plan_rejects ORDER BY reject_reason"))

Quarantined rows - the list that goes back to the submitting market


,submission_batch,submitted_country,plan_month,currency_code,plan_net_sales_local,reject_reason
0,FY2023-PLAN-V1,France,2023-08-01,EUR,NaN,Plan value is blank
1,FY2023-PLAN-V1,Japan,2023-11-01,JPY,NaN,Plan value is blank
2,FY2024-PLAN-V1,Australia,2024-02-01,AUD,NaN,Plan value is blank
3,FY2024-PLAN-V1,United Kingdom,2024-09-01,GBP,NaN,Plan value is blank
4,FY2023-PLAN-V1,Belgium,2023-05-01,EUR,-2685.19,Plan value is negative


In [29]:
print("End-to-end reconciliation")
display(q("""SELECT stream, stage, entity, row_count, ROUND(value_usd, 2) AS value_usd,
                    row_delta, expectation
             FROM qa.reconciliation_statement ORDER BY stream, step_order"""))

End-to-end reconciliation


,stream,stage,entity,row_count,value_usd,row_delta,expectation
0,Plan,Source file,regional_plan_submissions.csv,292,NaN,<NA>,Submitted in local currency - no USD value yet
1,Plan,stg,stg.plan_submissions,292,8078399.98,0,Harmonised and FX-converted - row count must equal source
2,Plan,qa,qa.plan_rejects,5,NaN,-287,"Quarantined rows, reported back to the submitting region"
3,Plan,core,core.fact_plan,287,8081299.98,282,Accepted rows only - source rows minus rejects
4,Sales,Source file,order_items.csv,180862,10796335.70,<NA>,Baseline: rows and gross value as delivered
5,Sales,stg,stg.order_items,180862,10796335.70,0,Typing and renaming only - must equal source
6,Sales,core,core.fact_order_item,180862,10796335.70,0,Joins to product and customer - must not drop rows
7,Sales,mart,mart.actual_country_month,135749,10796335.70,-45113,Aggregated to country x month - gross value must still tie


> **Conclusion.** **292 rows submitted, 5 quarantined, 287 accepted**, and the money ties at every stage. The one that matters commercially is Belgium: a single row submitted at -2,685.19 was subtracting from the consolidated plan, making the group budget 2,900 dollars easier to beat. A sign error does not announce itself - it just quietly flatters every achievement percentage computed against it.


---

## Part 14 - Customer concentration and segmentation

**What it does.** Two questions a commercial team asks about its customer base: how
concentrated is spend, and does the base split into groups worth treating differently.

The Gini coefficient measures concentration on a 0 to 1 scale - 0 is perfectly even
spend across all customers, 1 is a single customer holding everything. RFM segmentation
clusters customers on recency, frequency and monetary value, with k chosen on silhouette
score rather than assumed.

| | |
|---|---|
| **Built by** | `src/advanced_analytics.py` |
| **Written to** | `artifacts/advanced_metrics.json`, `data/processed/advanced/rfm_segments.csv` |


In [30]:
# The segmentation as the pipeline runs it - k is chosen on silhouette score, not assumed.
show_py("advanced_analytics.py", "rfm_segmentation", lines=45)

--- src/advanced_analytics.py - def rfm_segmentation - 118 lines ----------------
def rfm_segmentation(
    connection: duckdb.DuckDBPyConnection, output_dir: Path, figure_dir: Path
) -> dict[str, object]:
    customers = connection.execute(
        """
        SELECT customer_id, recency_days, order_count, net_sales_value,
               net_profit_value, observed_return_rate, country, acquisition_source
        FROM mart.customer_360
        WHERE net_sales_value > 0 AND order_count > 0
        """
    ).fetchdf()
    model_data = pd.DataFrame(
        {
            "recency": np.log1p(customers["recency_days"].clip(lower=0)),
            "frequency": np.log1p(customers["order_count"]),
            "monetary": np.log1p(customers["net_sales_value"]),
        }
    )
    scaler = RobustScaler()
    scaled = scaler.fit_transform(model_data)

    diagnostics: list[dict[str, float]] = []
    sample_size = min(12000, len(customers))
    rng = np.random.default_rng(RANDOM_STATE)
    sample_

In [31]:
# Gini coefficient on customer net spend.
# Population: everyone who placed an order. Customers whose orders all cancelled
# or returned net to zero are KEPT - they are real customers who contributed
# nothing, and excluding them would understate concentration.
spend = q("""SELECT customer_id, SUM(net_sales_value) AS net_spend
             FROM core.fact_order_item GROUP BY 1
             HAVING SUM(net_sales_value) >= 0""")
values = np.sort(spend.net_spend.to_numpy())
n = len(values)
gini = (2 * np.arange(1, n + 1) - n - 1).dot(values) / (n * values.sum())

print(f"Customers who placed an order : {n:,}")
print(f"  of whom net spend is zero   : {(values == 0).sum():,} ({(values == 0).mean():.1%})")
print(f"Gini coefficient              : {gini:.3f}")
print(f"Top 10% share of net revenue  : {values[int(n * 0.9):].sum() / values.sum():.1%}")
print(f"Top 20% share of net revenue  : {values[int(n * 0.8):].sum() / values.sum():.1%}")

# Sensitivity: the same measure on contributing customers only.
pos = values[values > 0]
m = len(pos)
gini_pos = (2 * np.arange(1, m + 1) - m - 1).dot(pos) / (m * pos.sum())
print(f"Excluding zero-spend customers, Gini falls to {gini_pos:.3f} on {m:,} customers.")

Customers who placed an order : 80,095
  of whom net spend is zero   : 13,880 (17.3%)
Gini coefficient              : 0.580
Top 10% share of net revenue  : 38.6%
Top 20% share of net revenue  : 58.7%
Excluding zero-spend customers, Gini falls to 0.491 on 66,215 customers.


In [32]:
# RFM segments, as computed by the pipeline.
metrics = json.loads((ROOT / "artifacts" / "advanced_metrics.json").read_text(encoding="utf-8"))
rfm = pd.DataFrame(metrics["rfm_segmentation"]["segments"])
rfm["revenue_share"] = rfm.total_net_sales / rfm.total_net_sales.sum()
rfm["customer_share"] = rfm.customers / rfm.customers.sum()
display(rfm[["segment", "customers", "customer_share", "median_recency_days",
             "mean_order_count", "median_net_sales", "total_net_sales",
             "revenue_share"]].round(3))
sel = metrics["rfm_segmentation"]
print(f"k selected on silhouette score: k={sel['selected_k']}, "
      f"score={sel['silhouette_selected_k']:.3f}")

,segment,customers,customer_share,median_recency_days,mean_order_count,median_net_sales,total_net_sales,revenue_share
0,Champions,19697,0.297,192.0,2.721,175.32,4285073.074,0.529
1,New / Developing,13007,0.196,32.0,1.391,63.00,1209493.061,0.149
2,Hibernating,33511,0.506,504.0,1.127,50.99,2601965.343,0.321


k selected on silhouette score: k=3, score=0.344


![RFM segments - size against value](../artifacts/figures/rfm_segments.png)

*RFM segments - size against value*


> **Conclusion.** Spend is **concentrated** - a Gini of 0.58, with the top 10% of customers holding 39% of net revenue and the top 20% holding 59%. A second finding sits inside that number: **17% of customers who placed an order contributed zero net revenue**, because everything they ordered was cancelled or returned. Excluding them drops the Gini to 0.49, which is why the population is stated rather than assumed. The segmentation splits 66,215 customers into three groups: **Champions are 30% of the base and 53% of revenue**, Hibernating are 51% of the base and 32% of revenue, and New/Developing are the remainder. The commercial implication is that retention spend targeted at Champions protects more revenue per dollar than broad reactivation of the Hibernating group.


---

## Part 15 - Forecasting - and the audit that released it

**What it does.** Forecasts monthly net sales to the end of the plan year, picks the
method by rolling-origin backtest rather than in-sample fit, allocates the group number
to markets on trailing share, and compares the result against plan.

This module sat under *Next steps* for a long time. It was written and it ran, but it
had not been read line by line, and a number I cannot defend undoes the point of
everything before it. What follows is the audit that changed that.

| | |
|---|---|
| **Built by** | `src/forecasting.py` |
| **Written to** | `mart.forecast_monthly`, `mart.forecast_accuracy`, `mart.forecast_backtest_detail`, `mart.forecast_scenarios`, `mart.forecast_country_month`, `mart.forecast_vs_plan` |


### The three candidates

| Method | What it assumes |
|---|---|
| `seasonal_naive` | next January looks like last January. The benchmark any method has to beat before it earns its complexity |
| `seasonal_naive_drift` | the same, scaled by the latest year-on-year movement. Captures trend without fitting anything |
| `holt_winters_log` | additive trend and seasonality fitted on log net sales, which is multiplicative on the original scale |

Deliberately small and deliberately conventional. A planning committee has to be able to
follow the method, and whoever inherits the project has to be able to rerun it.


In [33]:
scores = db.execute("SELECT * FROM mart.forecast_accuracy ORDER BY mape").fetchdf()
for col in ["mape", "mape_first_step", "mape_final_step", "worst_month_ape", "bias"]:
    scores[col] = (scores[col] * 100).round(1).astype(str) + "%"
scores

,model,mape,mape_first_step,mape_final_step,worst_month_ape,bias,horizon,origins,folds
0,seasonal_naive_drift,8.0%,6.5%,11.5%,39.4%,-0.7%,8,24,192
1,holt_winters_log,26.0%,8.3%,43.5%,163.1%,8.3%,8,24,192
2,seasonal_naive,42.4%,41.7%,44.2%,64.9%,-42.4%,8,24,192


### The defect this audit found

The backtest scored every model **one month ahead**. The production forecast then ran
the winner **eight months ahead**.

Forecast error grows with horizon, and not at the same rate for every method. A rule
that carries last year's shape forward degrades slowly; a fitted trend compounds its
trend error every step. Scoring at step one measures the first of those and says
nothing about the second.

`mart.forecast_backtest_detail` holds every fold, so the decay is queryable rather than
a claim:


In [34]:
decay = db.execute("""
    SELECT step,
           ROUND(AVG(CASE WHEN model = 'seasonal_naive_drift'
                     THEN absolute_percentage_error END) * 100, 1) AS drift_pct,
           ROUND(AVG(CASE WHEN model = 'holt_winters_log'
                     THEN absolute_percentage_error END) * 100, 1) AS holt_winters_pct,
           ROUND(AVG(CASE WHEN model = 'seasonal_naive'
                     THEN absolute_percentage_error END) * 100, 1) AS benchmark_pct,
           COUNT(*) AS folds
    FROM mart.forecast_backtest_detail
    GROUP BY step ORDER BY step
""").fetchdf()
decay

,step,drift_pct,holt_winters_pct,benchmark_pct,folds
0,1,6.5,8.3,41.7,72
1,2,6.6,13.5,41.6,72
2,3,5.9,17.5,41.9,72
3,4,7.9,22.3,42.1,72
4,5,7.3,29.0,42.1,72
5,6,8.0,34.7,42.7,72
6,7,10.3,39.0,43.1,72
7,8,11.5,43.5,44.2,72


At one month the two serious candidates look comparable. By month eight they are not -
and month eight is the one the plan conversation is actually about.

**The honest version.** The original design did not pick the wrong model. It picked the
right one with almost no evidence: over its six one-step folds the two candidates split
them three-three. It could as easily have landed on the other.

**The fix.** `backtest()` now takes the production horizon and scores every step at
every origin - 24 origins x 8 steps = 192 folds instead of 6 - and `run()` computes the
horizon first so the two can no longer drift apart.


In [35]:
show_py("forecasting.py", "backtest", lines=40)

--- src/forecasting.py - def backtest - 67 lines ----------------
def backtest(series: pd.Series, horizon: int = 1) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rolling-origin backtest scored across the whole forecast horizon.

    Returns (scores, detail). `scores` is one row per model; `detail` is one row
    per model x origin x step, written to mart.forecast_backtest_detail so the
    horizon decay is inspectable rather than taken on trust.

    For each origin the model is refitted on history up to that origin and asked
    for `horizon` months. Every step is scored, so a model that is sharp next
    month but drifts by month eight is penalised for the drift - which is the
    behaviour that matters when the forecast is used to call the full-year
    landing.
    """
    # offset counts back from the end. It must be >= horizon so every step of
    # the forecast has an actual to compare against, and small enough to leave
    # MIN_TRAIN_MONTHS of history for the fit.
    largest_o

### Six more checks, specified before they were run

| # | Check | Result |
|---|---|---|
| 2 | Does `exp(mean of logs)` bias Holt-Winters low? | **rejected** - measured bias +0.6%, negligible on this series |
| 3 | Does MAPE's asymmetry distort the ranking? | **rejected** - MAPE, sMAPE, RMSE% and median APE all rank identically |
| 4 | Is the ranking stable, or an artefact of six folds? | **defect** - re-scored over 32 origins the champion wins 27 of 27 windows, but the code had not established that |
| 5 | Does the drift clip `[0.5, 2.5]` ever bind? | **pass** - raw ratio ranged 1.54 to 2.39, clip bound 0 times |
| 6 | Is the incomplete final month excluded? | **pass** - `WHERE is_complete_month = 1`, May 2024 held out |
| 7 | Does the country allocation sum back to the total? | **pass** - max delta 0.000000 |

Two of the three most interesting hypotheses turned out to be wrong. That is worth
saying out loud: a review that confirms everything it suspected is not a review.

The full write-up is `docs/forecast_review.md`.


In [36]:
# Check 7, re-run here rather than asserted.
db.execute("""
    SELECT f.month_start,
           ROUND(f.forecast_net_sales_usd, 2) AS total,
           ROUND(a.allocated, 2)              AS allocated_to_markets,
           ROUND(a.allocated - f.forecast_net_sales_usd, 6) AS delta
    FROM mart.forecast_monthly f
    JOIN (SELECT month_start, SUM(forecast_net_sales_usd) AS allocated
          FROM mart.forecast_country_month GROUP BY 1) a USING (month_start)
    ORDER BY f.month_start
""").fetchdf()

,month_start,total,allocated_to_markets,delta
0,2024-05-01,524060.85,524060.85,0.0
1,2024-06-01,513758.10,513758.10,0.0
2,2024-07-01,581944.10,581944.10,-0.0
3,2024-08-01,615324.43,615324.43,0.0
4,2024-09-01,652846.30,652846.30,0.0
5,2024-10-01,699954.05,699954.05,0.0
6,2024-11-01,746169.20,746169.20,0.0
7,2024-12-01,815754.15,815754.15,-0.0


### The forecast itself, against plan


In [37]:
db.execute("""
    SELECT month_start, basis,
           ROUND(expected_net_sales_usd) AS expected,
           ROUND(plan_net_sales_usd)     AS plan,
           ROUND(gap_to_plan_usd)        AS gap,
           ROUND(expected_achievement_rate * 100, 1) AS achievement_pct
    FROM mart.forecast_vs_plan
    WHERE plan_net_sales_usd IS NOT NULL
    ORDER BY month_start
""").fetchdf()

,month_start,basis,expected,plan,gap,achievement_pct
0,2023-01-01,Actual,188680.0,204000.0,-15320.0,92.5
1,2023-02-01,Actual,167410.0,177000.0,-9590.0,94.6
2,2023-03-01,Actual,184778.0,214900.0,-30122.0,86.0
3,2023-04-01,Actual,193150.0,204900.0,-11750.0,94.3
4,2023-05-01,Actual,209624.0,212400.0,-2776.0,98.7
5,2023-06-01,Actual,205503.0,240000.0,-34497.0,85.6
6,2023-07-01,Actual,232778.0,248500.0,-15722.0,93.7
7,2023-08-01,Actual,246130.0,245300.0,830.0,100.3
8,2023-09-01,Actual,261139.0,248500.0,12639.0,105.1
9,2023-10-01,Actual,279982.0,275900.0,4082.0,101.5


### Two limitations that survive the audit

**TheLook is synthetic.** Its growth curve is far smoother than real demand, which
carries promotional spikes, stockouts and competitive shocks. An 8.0% MAPE here
demonstrates a validation method; it is not an accuracy claim that transfers to a real
business.

**The plan is constructed.** TheLook ships no budget data, so `src/build_plan_source.py`
generates it from a published deterministic rule. `mart.forecast_vs_plan` therefore
compares a real forecast against a synthetic target, and the achievement rate above
inherits that.


> **Conclusion.** The module was held back from every presented surface until it had been audited to the standard of the warehouse, and the audit found two things worth fixing and rejected two hypotheses that sounded right. That sequence - hold it back, specify the checks, publish what the checks found including the ones that found nothing - is the part of this section worth more than the forecast.


---

## Part 16 - Where the outputs go

**What it does.** Everything the pipeline produces lands in one of four places, and
every one of them is regenerated on each run so nothing can drift from the warehouse.

| | |
|---|---|
| **Built by** | `src/render_outputs.py`, `src/render_commercial_figures.py`, `src/build_excel_pack.py` |
| **Written to** | see the table below |


| Destination | Contents | Consumed by |
|---|---|---|
| `data/processed/power_bi/` | 34 CSV and Parquet exports | the Power BI report |
| `data/processed/analysis/` | 23 analysis-ready CSVs | ad-hoc analysis, this notebook's figures |
| `outputs/qa/` | 6 QA evidence files | reviewers, the validation gate |
| `artifacts/figures/` | 12 rendered charts | the README and this notebook |
| `reports/` | the Excel management pack | commercial colleagues who do not open Power BI |

### On the Excel pack

The Power BI report is the self-service surface and this notebook is the technical
walkthrough, so the Excel workbook is neither of those - it is the **distribution**
artifact. It is what circulates by email in a commercial team, where most readers will
never open a BI tool. It is regenerated from the warehouse on every run, so it cannot
drift from the dashboard, and it is one file rather than a folder of CSVs. It is not
needed to understand the project, which is why it is named here rather than rebuilt in
this notebook.


In [38]:
for folder in ["data/processed/power_bi/csv", "data/processed/analysis",
               "outputs/qa", "artifacts/figures", "reports"]:
    path = ROOT / folder
    if path.exists():
        files = sorted(p.name for p in path.iterdir() if p.is_file())
        print(f"{folder:<34} {len(files):>3} files")
    else:
        print(f"{folder:<34}   - not present")

data/processed/power_bi/csv         40 files
data/processed/analysis             24 files
outputs/qa                           6 files
artifacts/figures                   12 files
reports                              1 files


In [39]:
# The validation gate - the last thing the pipeline runs before it reports success.
report = json.loads((ROOT / "artifacts" / "validation_report.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=2)[:1200])

{
  "status": "PASS",
  "errors": []
}


> **Conclusion.** Every number in the dashboard, the Excel pack and this notebook comes from the same warehouse, regenerated by the same command. There is no manual step between the source CSVs and any published figure, which is what makes the whole thing reproducible rather than merely documented.


---

## Part 17 - Next steps

Three pieces of work that are **not finished**, named here rather than shown. No
results from any of them appear in this walkthrough, the dashboard or the management
pack - they are scoped, not delivered.


### 17.1 Return-propensity model - needs better features, not a better algorithm

A logistic regression on price, margin, category and department, trained on pre-2023
line items and tested on 2023 onward.

**Built by** `src/advanced_analytics.py` - **Result in** `artifacts/advanced_metrics.json`


The model was trained and scored, and the result is an **ROC AUC of 0.50** - a coin
flip. The numbers are in `artifacts/advanced_metrics.json`; they are not reproduced
here, because a model this project does not stand behind should not be given the
visual weight of a result.


**Next step.** The recommendation is to instrument better outcome logging - capture a
return reason code and size/fit at the point of return - rather than to try another
algorithm on the same features. Deploying a 0.50 AUC model would create work without
creating value.


### 17.2 Customer lifetime value - not attempted

CLV is the natural extension of the RFM work in Part 14, and it is deliberately absent.

**Next step.** A BG/NBD plus Gamma-Gamma model on the transaction history would give a
forward-looking value per customer, which turns the segmentation from descriptive into
actionable - it tells you what a Champion is worth over the next twelve months rather
than what they have been worth historically. The data supports it; the time did not.


---

## What this project demonstrates

| | |
|---|---|
| **Data engineering** | a layered warehouse, ten SQL modules run in sequence, idempotent and rebuildable from the seven source CSVs by one command |
| **Integration** | two independent source systems reconciled onto a conformed country-month grain, with FX conversion at a fixed budget rate |
| **Master data** | a genuine defect found in the source - 15 spellings for 13 markets - resolved with a committed country master and a gate that catches the sixteenth |
| **Data quality** | 35 checks across three gates, with a stated rule for what blocks a build and what quarantines and warns |
| **Reconciliation** | a statement a reviewer can audit without reading the code: 292 submitted, 5 quarantined, 287 accepted, money tying at every stage |
| **Commercial analysis** | plan-versus-actual variance, a price-volume-mix bridge that reconciles to the cent, and a monthly scorecard |
| **Forecasting** | three methods compared on 192 rolling-origin folds at the horizon actually used, with the audit that found two defects in the first version published alongside |
| **Honesty** | a failed model reported as failed, a flat margin named as a dataset property, a constructed plan file with its construction rule published, and two forecast hypotheses tested and reported as wrong |

### Limitations, stated

- The planning file is **constructed**, from a published deterministic rule. No budget ships with TheLook.
- Net margin is flat at 51.8% every month, which is a property of how the dataset was generated, not a commercial finding.
- There is no price variation in the source, so the price elasticity work is method rather than evidence.
- The return model failed, and the recommendation is better logging rather than a better algorithm.
- The forecast MAPE is optimistic: synthetic growth is smoother than real demand, and the plan it
  is measured against is constructed rather than supplied.

---

*Full written analysis: `docs/analysis_and_findings.md` and `docs/commercial_analytics.md`.*
*Technical reference: `docs/technical_reference.md`. Metric definitions: `docs/kpi_dictionary.md`.*


In [40]:
db.close()
print("Walkthrough complete. The warehouse connection is closed.")

Walkthrough complete. The warehouse connection is closed.